# Spielman et al. (1993): Transmission/Disequilibrium Test

In Spielman et al., the Transmission/Disequilibrium Test (TDT) is introduced to assess whether a genetic marker results in a population-level association. This design works by restricting to heterozygous parents, who have one copy of each allele. Under no association, each allele should be equally likely to be transmitted to offspring. If an allele was instead associated with disease, then transmission to affected children should happen more often than chance. Spielman et al. apply this approach is the context of  Type 1 diabetes (insulin-dependent diabetes mellitus). Here, we re-examine their application using estimating equations

## Setup

In [1]:
import numpy as np
import scipy as sp
import pandas as pd
import delicatessen as deli
from delicatessen import MEstimator
from delicatessen.utilities import inverse_logit, aggregate_efuncs

print("Versions")
print("NumPy:       ", np.__version__)
print("SciPy:       ", sp.__version__)
print("Pandas:      ", pd.__version__)
print("Delicatessen:", deli.__version__)

Versions
NumPy:        2.3.5
SciPy:        1.16.3
Pandas:       2.3.3
Delicatessen: 4.2


In [2]:
y = np.asarray([1, ]*78 + [0, ]*46)

The corresponding data is available from Table 5 of their paper. In particular, there are 78 of 124 disease-associated alleles transmitted. To apply the TDT method, this data is sufficient.

The TDT compares the proportion of transmitted candidate allele versus the expected. Here, there are only two alleles, so the expected transmission is 0.50. TDT compares the proportion observed in the data to this expectation.

For estimating equations, we simply need to estimate the proportion of transmitted alleles. Then we can use the sandwich variance estimator for inference. There are several ways to estimate the proportion. The simplest is to use the following estimating function
$$ \psi(O_i; \rho) = Y_i - \rho $$
where $\rho$ is the proportion and $Y_i$ is an indicator for the candidate allele. The following code applies this estimator

In [3]:
def psi(theta):
    return y - theta[0]

In [4]:
estr = MEstimator(psi, init=[0.5])
estr.estimate()

Now we can conduct the statistical test comparing the observed proportion to the expected proportion. There are several ways to do this. The easiest is to compute the corresponding P-value. The following code applies this process

Then we can aslo....

In [5]:
estr.p_values(null=0.5)

array([0.00293528])

This P-value indicates the allele transmission is quite different from the expectation of 0.5, which would then indicate that this allele is related to Type 1 Diabetes.

Rather than a P-value, we can also compute S-values, which is the 'surprisal' value. It translates the P-value into how many heads in a row (with a fair coin) would need to occur. This conversion can be done in `delicatessen`

In [6]:
estr.s_values(null=0.5)

array([8.41228684])

So, the comparison we see is as unlikely as getting more than 8 heads in a row with a fair coin. That's pretty unlikely (if you don't think so, try flipping a coin until you get 8 heads in a row).

As a final option, we can also look at the confidence intervals. 

In [7]:
estr.confidence_intervals()

array([[0.54400821, 0.71405631]])

The confidence intervals are quite wide, but are fairly above 0.5.

While not a concern here, the previous estimating function can have poor performance when the true $\rho$ is near the boundaries of the parameter space. To ensure our estimate is bounded, we can modify the previous estimating function so that it is bounded in the parameter space. We do this via the inverse-logistic function

In [8]:
def psi(theta):
    return y - inverse_logit(theta[0])

In [9]:
estr = MEstimator(psi, init=[0.])
estr.estimate()

In [10]:
estr.p_values(null=0.)

array([0.00450337])

In [11]:
estr.s_values(null=0.)

array([7.79478032])

In [12]:
inverse_logit(estr.confidence_intervals())

array([[0.54083528, 0.7093912 ]])

These results are quite similar to the previous. However, note that we are now testing whether the transformed parameter is equal to zero (since $\text{logit}(0.5) = 0$. This is generally to be expected in this example since the proportion seems to be 'far' from the boundary points of 0 or 1.

## Clustering

An issue not discussed in Spielman et al. is the impact of potential clustering. These alleles came from 57 parents. Since some parents had multiple children, we might be concerned about clustering by parents. In particular, the previous data does not constitute completely independent observations. As such, the effective sample size is likely lower than the 124 from before. Therefore, the variance estimate is likely too small (and thus our statistical inference is not correct). TDT does not handle this clustering naturally, but estimating equations provide an easy solution to this problem.

Unfortunately, Spielman et al. does not indicate which data came from the same parents. Therefore, we instead simulate a group variable. This following analysis should be viewed as illustrative of the approach and should not be interpreted.

In [13]:
np.random.seed(123454321)
family = list(range(1, 58)) + list(range(1, 58)) + [1, 1, 1, 1, 2, 2, 3, 3, 4, 4]
np.random.shuffle(family)

In [14]:
def psi_individual(theta):
    return y - inverse_logit(theta[0])

In [15]:
def psi_cluster(theta):
    return aggregate_efuncs(psi_individual(theta), group=family)

In [16]:
estr = MEstimator(psi_cluster, init=[0.])
estr.estimate()

In [17]:
estr.p_values(null=0.)

array([0.00361566])

In [18]:
estr.s_values(null=0.)

array([8.11152533])

In [19]:
inverse_logit(estr.confidence_intervals())

array([[0.54298989, 0.70759864]])

Here, the cluster-adjusted analysis did not change the results much. This is to be expected since we cannot link the alleles together and randomly assigned groups here. In practice, clustering may make a bigger impact on statistical inference.

## References

Spielman RS, McGinnis RE, & Ewens WJ (1993). Transmission test for linkage disequilibrium: the insulin gene region and insulin-dependent diabetes mellitus (IDDM). *American Journal of Human Genetics*, 52(3), 506.